**Imports and setup** for the calibration workflow: `spectral`, scikit-learn, matplotlib, and the interactive **hsiViewer** ROI tools.

In [ ]:
from sklearn import linear_model
import matplotlib.pyplot as plt
from matplotlib import colors
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.decomposition import PCA
import numpy as np
from sklearn.mixture import GaussianMixture
import numpy as np
import copy
import spectral
import time
import csv
import os
import importlib
import pickle
from hsiViewer import hsi_viewer_layers as hlv
from hsiViewer import hsi_viewer_ROI as hvr
import matplotlib as mpl
mpl.rcParams['lines.linewidth'] = 0.75

# --- Load configuration (paths + parameters live in config.yaml) ---
# config.yaml and the paths inside it are relative to the repo root, but this
# notebook lives in notebooks/. Walk up to the repo root and resolve every
# configured path against it, so this works whether Jupyter is launched from
# the repo root or from notebooks/.
import yaml
from pathlib import Path
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'config.yaml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
with open(REPO_ROOT / 'config.yaml') as _f:
    CONFIG = yaml.safe_load(_f)
for _section in ('paths',):
    for _key, _val in CONFIG.get(_section, {}).items():
        if isinstance(_val, str):
            CONFIG[_section][_key] = str(REPO_ROOT / _val)
# Notebook 01 writes its calibration bundle (gain, offset, and the panel
# spectra intermediates) into calibration_dir -- gitignored and absent in a
# fresh clone -- so create it. Each collection has its own calibration_dir, so
# a re-run never clobbers another collection's coefficients.
os.makedirs(CONFIG['paths']['calibration_dir'], exist_ok=True)

## 1. Open the Image with the Calibration Panel

**Open the cal-panel image.** Reads the raw image that contains the calibration tarps and its wavelengths. Set this image in `config.yaml`.

In [ ]:
# Raw image containing the calibration panels (set both entries in config.yaml).
cal_image_hdr = CONFIG['paths']['cal_image_hdr']
cal_image = CONFIG['paths']['cal_image']
assert Path(cal_image_hdr).stem == Path(cal_image).stem, \
    "cal_image and cal_image_hdr name different cubes — check config.yaml"

# Read the image
im = spectral.envi.open(cal_image_hdr, cal_image)
im.Arr = im.load().astype(np.float32)
im.mask = im.Arr[:,:,0]!=0
nr, nc, nb = im.Arr.shape
im.wl = np.asarray(im.bands.centers)
wl = im.wl

**Load and resample the tarp library.** Loads the ASD cal-tarp reference spectra and resamples them to the image's bands, so measured and reference tarp spectra share one band axis.

In [ ]:
# Spectral Libraries for Cal Tarps
fname_sli = CONFIG['paths']['cal_library_sli']
fname_sli_hdr = CONFIG['paths']['cal_library_hdr']
lib = spectral.envi.open(fname_sli_hdr, fname_sli)
ns, nb_lib = lib.spectra.shape
lib.wl = np.asarray(lib.bands.centers)
wl_lib = lib.wl
print(f'nSpectra, initial nBands = {lib.spectra.shape}')
# resample library to image
resampler = spectral.BandResampler(lib.wl, im.wl)
spectra = resampler(lib.spectra.T).T
print(f'nSpectra, nBands = {spectra.shape}')

## 2. Create ROIs on the Calibration Panel

<p style="color:red">Run the cell with hrv.viewer to select the regions of interest on the low and mid panels.</p>

**Interactive.** Opens the hsiViewer ROI tool — draw regions of interest on the low and mid reflectance tarps and save them (loaded in the next cell).

In [ ]:
hvr.viewer(im, stretch=[2,100])

**Load the panel ROIs.** Reads the saved cal-panel ROIs and extracts the raw spectra for the low and mid panels.

In [ ]:
# Read Cal Panel ROIs and create panel_low_spectra and panel_mid_spectra
with open(CONFIG['paths']['cal_panel_rois'], 'rb') as f:
    cal_panel_rois = pickle.load(f)
# Print the names of the ROIs
names = cal_panel_rois.names
print(f'ROI Names: {names}')
# Get the spectra for the panels
condition = cal_panel_rois.df['Name'] == 'Cal Panel Mid'
df_subset_age = cal_panel_rois.df[condition]
panel_mid_spectra = np.asarray(df_subset_age.iloc[:,4:])
condition = cal_panel_rois.df['Name'] == 'Cal Panel Low'
df_subset_age = cal_panel_rois.df[condition]
panel_low_spectra = np.asarray(df_subset_age.iloc[:,4:])

**Drop saturated pixels** (near the sensor's maximum) so they don't bias the calibration fit.

In [ ]:
# Remove Saturated Pixels
print(f'Number of spectra before removing saturated poixels: low={panel_low_spectra.shape[0]}, mid={panel_mid_spectra.shape[0]}')
saturation_trheshold = int(0.97*np.max(panel_mid_spectra))
panel_low_spectra = panel_low_spectra[np.max(panel_low_spectra, axis=1) < saturation_trheshold, :]
panel_mid_spectra = panel_mid_spectra[np.max(panel_mid_spectra, axis=1) < saturation_trheshold, :]
print(f'Number of spectra after removing saturated poixels: low={panel_low_spectra.shape[0]}, mid={panel_mid_spectra.shape[0]}')

**Average** the low and mid panel spectra to one mean spectrum each.

In [ ]:
# Calculate Means
panel_low_mean = np.mean(panel_low_spectra, axis=0)
panel_mid_mean = np.mean(panel_mid_spectra, axis=0)

**Plot** the raw and normalized panel spectra to visually confirm the ROIs are clean.

In [ ]:
plt.figure(figsize=(20, 8))
plt.subplot(1, 2, 1)
for i in range(panel_low_spectra.shape[0]):
    plt.plot(im.wl, panel_low_spectra[i,:], lw=0.5)
plt.plot(im.wl, panel_low_mean, lw=3, c='lime')
plt.grid(True)
plt.xlabel('Wavelength (nm)')
plt.ylabel('Count')
plt.title('Raw Data Values, Low Reflectance Tarp Area');

plt.subplot(1, 2, 2)
for i in range(panel_mid_spectra.shape[0]):
    plt.plot(im.wl, panel_mid_spectra[i,:], lw=0.5)
plt.plot(im.wl, panel_mid_mean, lw=3, c='red')
plt.grid(True)
plt.xlabel('Wavelength (nm)')
plt.ylabel('Count')
plt.title('Raw Data Values, Medium Reflectance Tarp Area');

plt.figure(figsize=(20,8))
plt.subplot(1, 2, 1)
for i in range(panel_low_spectra.shape[0]):
    plt.plot(im.wl, panel_low_spectra[i,:]/np.mean(panel_low_spectra[i,:]), lw=0.5)
plt.plot(im.wl, panel_low_mean/np.mean(panel_low_mean), lw=3, c='lime')
plt.grid(True)
plt.xlabel('Wavelength (nm)')
plt.ylabel('Count')
plt.title('Normalized Data Values, Low Reflectance Tarp Area');

plt.subplot(1, 2, 2)
for i in range(panel_mid_spectra.shape[0]):
    plt.plot(im.wl, panel_mid_spectra[i,:]/np.mean(panel_mid_spectra[i,:]), lw=0.5)
plt.plot(im.wl, panel_mid_mean/np.mean(panel_mid_mean), lw=3, c='red')
plt.grid(True)
plt.xlabel('Wavelength (nm)')
plt.ylabel('Count')
plt.title('Normalized Data Values, Medium Reflectance Tarp Area');

**Save** the panel spectra for reference/reuse.

In [ ]:
# Save the panel-spectra intermediates into this collection's calibration_dir.
import os
_cal_dir = CONFIG['paths']['calibration_dir']
np.save(os.path.join(_cal_dir, 'panel_low_spectra.npy'), panel_low_spectra)
np.save(os.path.join(_cal_dir, 'panel_mid_spectra.npy'), panel_mid_spectra)

## 3. Calculate the Gain and Offset

**Gather ASD reference tarps.** Pulls the high / mid / dark tarp spectra out of the resampled library and averages each.

In [ ]:
# build high-ref tarp spectra
idx_h = []
idx_m = []
idx_l = []
for i,n in enumerate(lib.names):
    if 'high' in n:
        idx_h.append(i)
    if 'med' in n:
        idx_m.append(i)
    if 'dark' in n:
        idx_l.append(i)
        
asd_spc_h = np.zeros((len(idx_h), nb))
for i,j in enumerate(idx_h):
    asd_spc_h[i,:] = spectra[j,:]
asdhm = np.mean(asd_spc_h, axis=0)
    
asd_spc_m = np.zeros((len(idx_m), nb))
for i,j in enumerate(idx_m):
    asd_spc_m[i,:] = spectra[j,:]
asdmm = np.mean(asd_spc_m, axis=0)
    
asd_spc_l = np.zeros((len(idx_l), nb))
for i,j in enumerate(idx_l):
    asd_spc_l[i,:] = spectra[j,:]
asdlm = np.mean(asd_spc_l, axis=0)

**Plot** the ASD reference tarp spectra.

In [ ]:
plt.figure(figsize=(8, 6))
for i in range(asd_spc_h.shape[0]):
    plt.plot(im.wl, asd_spc_h[i,:], c='r')
plt.plot(im.wl, asdhm, c='r', lw=4, label='High_Ref_Panel')
for i in range(asd_spc_m.shape[0]):
    plt.plot(im.wl, asd_spc_m[i,:], c='g')
plt.plot(im.wl, asdmm, c='g', lw=4, label='Mid_Ref_Panel')
for i in range(asd_spc_l.shape[0]):
    plt.plot(im.wl, asd_spc_l[i,:], c='b')
plt.plot(im.wl, asdlm, c='b', lw=4, label='Low_Ref_Panel')
plt.grid(True)
plt.xlabel('Wavelength (nm)')
plt.ylabel('Reflectance')
plt.title('ASD Tarp Panel Spectra')
plt.legend()
plt.tight_layout()

**Single-band check.** Fits gain/offset for one band by regressing reference reflectance on measured counts.

In [ ]:
# Compute the gain for band 0
y = np.asarray([asdlm[0], asdmm[0]]).reshape(-1, 1)
X = np.asarray([panel_low_mean[0], panel_mid_mean[0]]).reshape(-1, 1)
reg = linear_model.LinearRegression(fit_intercept=True, n_jobs=-1)
reg.fit(X, y)
print(reg.intercept_[0])
print(reg.coef_[0][0])

**Visualize** the linear fit for one band index — the relationship the empirical-line calibration relies on.

In [ ]:
# Compute the gain for band idx
plt.figure(figsize=(6,6))
idx = 150 # band index
y = np.asarray([0, asdlm[idx], asdmm[idx]]).reshape(-1, 1)
X = np.asarray([0, panel_low_mean[idx], panel_mid_mean[idx]]).reshape(-1, 1)
reg = linear_model.LinearRegression(fit_intercept=True, n_jobs=-1)
reg.fit(X, y)
m = reg.coef_[0][0]
b = reg.intercept_[0]
plt.scatter(0, 0, c='m')
plt.scatter(panel_mid_mean[idx], asdmm[idx], c='g')
plt.scatter(panel_low_mean[idx], asdlm[idx], c='b')
xVals = np.linspace(0,panel_mid_mean[idx],50)
plt.plot(xVals,b+m*xVals)
plt.grid(True)
plt.xlabel('Wavelength (nm)')
plt.ylabel('Reflectance')
plt.title('ASD Spectra')
plt.tight_layout()

**Fit every band.** Empirical-line calibration: regress reference reflectance on measured panel counts for all bands, then plot the resulting gain and offset curves.

In [ ]:
offset = []
gain = []
use_all_regions = True
if use_all_regions:
    for i in range(len(im.wl)):
        y = np.asarray([0, asdlm[i], asdmm[i]]).reshape(-1, 1)
        X = np.asarray([0, panel_low_mean[i], panel_mid_mean[i]]).reshape(-1, 1)
        reg = linear_model.LinearRegression(fit_intercept=True, n_jobs=-1)
        reg.fit(X, y)
        gain.append(reg.coef_[0][0])
        offset.append(reg.intercept_[0])
else:
    for i in range(len(im.wl)):
        gain.append(asdhm[i]/thm[i])
        offset.append(0)
gain = np.asarray(gain, dtype=np.float32)
offset = np.asarray(offset, dtype=np.float32)
plt.plot(im.wl, gain, label='gain')
plt.plot(im.wl, offset, label='offset')
plt.legend()
plt.grid(True)

**Save the calibration.** Writes the per-band `gain` and `offset` that notebook 02 (and the batch script) apply to convert imagery to reflectance.

In [ ]:
# Save gain/offset into this collection's calibration_dir. Notebook 02 and the
# batch script read gain.npy/offset.npy from here for the same collection.
_cal_dir = CONFIG['paths']['calibration_dir']
np.save(os.path.join(_cal_dir, 'gain.npy'), gain)
np.save(os.path.join(_cal_dir, 'offset.npy'), offset)
print('Wrote calibration bundle for this collection to ' + _cal_dir)